<h1><center>Irish Temperature Analysis</center></h1>
<h2><center>Can we predict tomorrows temperatures based on past values?</center></h2>

<h3><center>A Report by Cathal O'D<h3></center>


This project compiles weather temperature data from 2021-2024 into a single dataset and then uses that data to do a prediction. 

Firstly, we import the usual libraries for this kind of analysis. However, compiling the temperature data into a single data set requires some work due to the way in which the the data itself is stored.

The data itself is downloaded from the Data Repository of the Irish government which is found at https://data.gov.ie Temperature specific data is available from this link: https://data.gov.ie/dataset/daily-air-temperatures-from-1961-2021 

Each month of each year is stored in a seperate CSV file. The naming convention is IRL_DLY_{xn}_{yyyy}{month}_grid_IDW_09Z.csv where {xn} determines the type of temperature value (High or Low), {yyyy} is for the year and {month} is for the month. For example, high temperature readings for January 2022 is contained in the file IRL_DLY_TX_202201_grid_IDW_09.csv. In order to compile 4 years worth of readings into a single dataset, it is necessary to design 2 functions. 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.statespace.sarimax import SARIMAX
pd.options.mode.chained_assignment = None  # default='warn'
from statsmodels.tools.sm_exceptions import  ValueWarning 

First, we have the 2 functions. The 1st function is designed to take in the CSV file, extract the location specific data and collapse it into rows. Firstly, let's import one of the files and have a look at it's structure
7	-2.7	-1


In [2]:
met_data_structure=pd.read_csv("met_data\IRL_DLY_TN_202101_grid_IDW_09Z.csv")
met_data_structure

,east,north,20210101,20210102,20210103,20210104,20210105,20210106,20210107,20210108,...,20210122,20210123,20210124,20210125,20210126,20210127,20210128,20210129,20210130,20210131
0,100000,100000,0.2,-2.7,-3.5,-3.3,-2.4,-3.6,-3.6,-2.1,...,-1.3,-3.7,-3.2,-4.5,-2.7,6.8,8.1,6.6,4.8,3.5
1,100000,101000,0.7,-2.2,-3.0,-2.8,-1.9,-3.1,-3.2,-1.6,...,-0.8,-3.2,-2.7,-4.0,-2.2,7.3,8.6,7.1,5.3,4.0
2,100000,102000,0.8,-2.3,-3.1,-2.9,-1.9,-3.1,-3.2,-1.6,...,-0.8,-3.2,-2.7,-4.0,-2.2,7.3,8.6,7.1,5.3,4.0
3,100000,103000,1.1,-2.0,-2.8,-2.6,-1.6,-2.8,-2.9,-1.3,...,-0.5,-2.9,-2.4,-3.7,-1.9,7.7,8.9,7.4,5.5,4.3
4,100000,104000,1.1,-2.0,-2.9,-2.7,-1.6,-2.8,-2.9,-1.3,...,-0.5,-2.9,-2.4,-3.6,-1.9,7.7,8.9,7.4,5.5,4.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70184,99000,95000,0.8,-2.0,-2.7,-2.6,-1.7,-2.9,-3.0,-1.5,...,-0.7,-3.1,-2.6,-3.9,-2.1,7.4,8.7,7.2,5.4,4.1
70185,99000,96000,0.8,-2.1,-2.9,-2.7,-1.8,-3.0,-3.1,-1.6,...,-0.8,-3.2,-2.7,-4.0,-2.2,7.3,8.6,7.1,5.3,4.0
70186,99000,97000,0.6,-2.4,-3.1,-2.9,-2.0,-3.2,-3.3,-1.8,...,-1.0,-3.4,-2.9,-4.2,-2.4,7.1,8.4,6.9,5.1,3.8
70187,99000,98000,0.2,-2.8,-3.5,-3.4,-2.4,-3.6,-3.7,-2.2,...,-1.4,-3.8,-3.3,-4.6,-2.8,6.7,8.0,6.5,4.7,3.4


The first two columns determine the location of the temperature reading using the eirgrid co-ordinate system. More information on this can be found via this link:

https://irish.gridreferencefinder.com/

In order to get location specific data, you feed the location you want into the finder and it returns east and north co-ordinates. The rest of the columns are each date of that month. In the sample above the data is from January 2021. Now let's look at the 2 functions.

In [3]:
def met_data_import(csv_file,east_coord,north_coord):
    met_data=pd.read_csv(csv_file) # reads the CSV file
    met_data_month=met_data[met_data['east']==east_coord] #takes in the east grid reference
    met_data_month=met_data_month[met_data_month['north']==north_coord]# takes in the north grid reference
    met_data_month_converted=met_data_month.melt(["east", "north"], 
                               var_name="Date",
                               value_name="Value") # collapses the date columns into rows and assigns the temperature reading
    met_data_month_converted["Temp_date"]=pd.to_datetime(met_data_month_converted["Date"], format="%Y%m%d") # converts date column to date/time format
    met_data_month_converted=met_data_month_converted.drop(["Date"],axis=1)
    return met_data_month_converted #returns the dataset


def met_data_import_2(yyyy,east_coord,north_coord,xn): # utilises the det_data_import function and takes in year and temperature type
    
    # this list is to assign each month a different data set
    months=["met_data_jan","met_data_feb","met_data_mar","met_data_apr","met_data_may","met_data_jun", 
                       "met_data_jul", "met_data_aug", "met_data_sep", "met_data_oct", "met_data_nov", "met_data_dec"]
    #This list is the month name variable in the filename
    mm=(["01","02","03","04","05","06","07","08","09","10","11","12"])
    #Initialises the dictionary
    met_data_dict = {}

    for i in range(12):
        month = mm[i]
        filename = f"met_data/IRL_DLY_{xn}_{yyyy}{month}_grid_IDW_09Z.csv" #reads the csv file into each specific dataset
        month_name = months[i] #increments the month location value
        met_data_dict[month_name] = met_data_import(filename, east_coord, north_coord) #uses the met_data_import function to read each csv file
    combined_df = pd.concat(met_data_dict.values(), ignore_index=True) #compiles all the different dataset into a single file for the year
    return combined_df
    


Now that the functions have been designed, we can feed in the datasets specific for the year and location. We'll start with Dublin. 

Dublin city centre has the eirgrid co-ordinates East:315000 North: 234000. For each year in our analysis, we feed in 2 csv files, one for highest recorded temperature and the other for lowest recorded temperature. After feeding all the necessary information into the functions, we then name each column and then compile each year into a single dataset for the year.

In [4]:
met_data_dublin_high_2021=met_data_import_2(2021,315000,234000,"TX") #imports dublin high temperature data from 2021
met_data_dublin_low_2021=met_data_import_2(2021,315000,234000,"TN") #imports dublin low temperature data from 2021
met_data_dublin_high_2022=met_data_import_2(2022,315000,234000,"TX")
met_data_dublin_low_2022=met_data_import_2(2022,315000,234000,"TN")
met_data_dublin_high_2023=met_data_import_2(2023,315000,234000,"TX")
met_data_dublin_low_2023=met_data_import_2(2023,315000,234000,"TN")
met_data_dublin_high_2024=met_data_import_2(2024,315000,234000,"TX")
met_data_dublin_low_2024=met_data_import_2(2024,315000,234000,"TN")
met_data_dublin_high_2021.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_dublin_low_2021.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_dublin_high_2022.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_dublin_low_2022.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_dublin_high_2023.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_dublin_low_2023.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_dublin_high_2024.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_dublin_low_2024.columns=(['East', 'North', 'Low', 'Temp_date'])

#Compile the datasets for each year into a single dataset.
met_data_dublin_high=pd.concat([met_data_dublin_high_2021,met_data_dublin_high_2022,met_data_dublin_high_2023,met_data_dublin_high_2024],join='outer')
met_data_dublin_low=pd.concat([met_data_dublin_low_2021,met_data_dublin_low_2022,met_data_dublin_low_2023,met_data_dublin_low_2024],join='outer')
met_data_dublin_low=met_data_dublin_low.drop(["East","North"],axis=1) # necessary to drop these columns before merge.
met_data_dublin=met_data_dublin_high.merge(met_data_dublin_low,left_on="Temp_date",right_on="Temp_date") # merge the "high" and "low" datasets
met_data_dublin['Location']="Dublin" #assigns an extra column for location
met_data_dublin

,East,North,High,Temp_date,Low,Location
0,315000,234000,5.4,2021-01-01,-0.2,Dublin
1,315000,234000,3.9,2021-01-02,-1.7,Dublin
2,315000,234000,5.4,2021-01-03,-1.1,Dublin
3,315000,234000,5.6,2021-01-04,-0.3,Dublin
4,315000,234000,5.1,2021-01-05,2.5,Dublin
...,...,...,...,...,...,...
1456,315000,234000,10.5,2024-12-27,7.0,Dublin
1457,315000,234000,10.3,2024-12-28,7.5,Dublin
1458,315000,234000,10.6,2024-12-29,7.3,Dublin
1459,315000,234000,11.1,2024-12-30,8.4,Dublin


Now we have a dataset for Dublin High and Low temperatures for each month from 2021 to 2024. Now we can compile temperature readings from Waterford and Letterkenny and fed them into relevant datasets.

In [5]:
met_data_waterford_high_2021=met_data_import_2(2021,260000,112000,"TX")
met_data_waterford_low_2021=met_data_import_2(2021,260000,112000,"TN")
met_data_waterford_high_2022=met_data_import_2(2022,260000,112000,"TX")
met_data_waterford_low_2022=met_data_import_2(2022,260000,112000,"TN")
met_data_waterford_high_2023=met_data_import_2(2023,260000,112000,"TX")
met_data_waterford_low_2023=met_data_import_2(2023,260000,112000,"TN")
met_data_waterford_high_2024=met_data_import_2(2024,260000,112000,"TX")
met_data_waterford_low_2024=met_data_import_2(2024,260000,112000,"TN")
met_data_waterford_high_2021.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_waterford_low_2021.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_waterford_high_2022.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_waterford_low_2022.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_waterford_high_2023.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_waterford_low_2023.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_waterford_high_2024.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_waterford_low_2024.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_waterford_high=pd.concat([met_data_waterford_high_2021,met_data_waterford_high_2022,met_data_waterford_high_2023,met_data_waterford_high_2024],join='outer')
met_data_waterford_low=pd.concat([met_data_waterford_low_2021,met_data_waterford_low_2022,met_data_waterford_low_2023,met_data_waterford_low_2024],join='outer')
met_data_waterford_low=met_data_waterford_low.drop(["East","North"],axis=1)
met_data_waterford=met_data_waterford_high.merge(met_data_waterford_low,left_on="Temp_date",right_on="Temp_date")
met_data_waterford['Location']="waterford"
met_data_waterford

,East,North,High,Temp_date,Low,Location
0,260000,112000,6.2,2021-01-01,0.9,waterford
1,260000,112000,5.4,2021-01-02,-0.6,waterford
2,260000,112000,3.8,2021-01-03,-0.8,waterford
3,260000,112000,5.1,2021-01-04,-0.6,waterford
4,260000,112000,5.0,2021-01-05,2.1,waterford
...,...,...,...,...,...,...
1456,260000,112000,9.1,2024-12-27,8.0,waterford
1457,260000,112000,9.0,2024-12-28,8.3,waterford
1458,260000,112000,10.9,2024-12-29,6.2,waterford
1459,260000,112000,11.1,2024-12-30,5.5,waterford


In [6]:
met_data_letterkenny_high_2021=met_data_import_2(2021,216000, 411000,"TX")
met_data_letterkenny_low_2021=met_data_import_2(2021,216000, 411000,"TN")
met_data_letterkenny_high_2022=met_data_import_2(2022,216000, 411000,"TX")
met_data_letterkenny_low_2022=met_data_import_2(2022,216000, 411000,"TN")
met_data_letterkenny_high_2023=met_data_import_2(2023,216000, 411000,"TX")
met_data_letterkenny_low_2023=met_data_import_2(2023,216000, 411000,"TN")
met_data_letterkenny_high_2024=met_data_import_2(2024,216000, 411000,"TX")
met_data_letterkenny_low_2024=met_data_import_2(2024,216000, 411000,"TN")
met_data_letterkenny_high_2021.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_letterkenny_low_2021.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_letterkenny_high_2022.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_letterkenny_low_2022.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_letterkenny_high_2023.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_letterkenny_low_2023.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_letterkenny_high_2024.columns=(['East', 'North', 'High', 'Temp_date'])
met_data_letterkenny_low_2024.columns=(['East', 'North', 'Low', 'Temp_date'])
met_data_letterkenny_high=pd.concat([met_data_letterkenny_high_2021,met_data_letterkenny_high_2022,met_data_letterkenny_high_2023,met_data_letterkenny_high_2024],join='outer')
met_data_letterkenny_low=pd.concat([met_data_letterkenny_low_2021,met_data_letterkenny_low_2022,met_data_letterkenny_low_2023,met_data_letterkenny_low_2024],join='outer')
met_data_letterkenny_low=met_data_letterkenny_low.drop(["East","North"],axis=1)
met_data_letterkenny=met_data_letterkenny_high.merge(met_data_letterkenny_low,left_on="Temp_date",right_on="Temp_date")
met_data_letterkenny['Location']="letterkenny"
met_data_letterkenny

,East,North,High,Temp_date,Low,Location
0,216000,411000,5.7,2021-01-01,2.5,letterkenny
1,216000,411000,4.9,2021-01-02,0.0,letterkenny
2,216000,411000,4.6,2021-01-03,-0.4,letterkenny
3,216000,411000,4.6,2021-01-04,-1.1,letterkenny
4,216000,411000,4.9,2021-01-05,-0.4,letterkenny
...,...,...,...,...,...,...
1456,216000,411000,11.2,2024-12-27,8.8,letterkenny
1457,216000,411000,11.3,2024-12-28,8.9,letterkenny
1458,216000,411000,10.6,2024-12-29,5.9,letterkenny
1459,216000,411000,12.3,2024-12-30,8.8,letterkenny


Now that we have fed all the data from 2021 to 2024 into data sets for each city, now we can do a prediction on each city. We'll start with Dublin and then do Waterford and Letterkenny.

We want the algorithm to predict the temperatures of the 1st 7 days of August 2024 and then compare the results with the actual temperatures taken for those days. We'll use the Sarima algorithm with a Linear Regression model.

In [7]:
# Load and prepare data
met_data_query=met_data_dublin.query("Temp_date<='2024-07-31'") # This algorithm uses whatever date the dataset ends on as it's starting point
met_data_query['Temp_date'] = pd.to_datetime(met_data_query['Temp_date']) # in order to do a prediction of August 2024, we need to select the
met_data_query = met_data_query.sort_values("Temp_date") # up to July 31st 2024

# Use day number as a feature
met_data_query['day_num'] = (met_data_query['Temp_date'] - met_data_query['Temp_date'].min()).dt.days

# Fit model
X = met_data_query[['day_num']]
y = met_data_query['High']
model = LinearRegression()
model.fit(X, y)

# Predict next 7 days
future_days = pd.DataFrame({'day_num': range(met_data_query['day_num'].max() + 1, met_data_query['day_num'].max() + 8)})
predictions = model.predict(future_days)

# Set datetime index
ts = met_data_query.set_index('Temp_date')['High']

# Fit model
model = SARIMAX(ts, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
results = model.fit()

# Forecast
forecast_steps = 7
forecast = results.forecast(steps=forecast_steps)

# Generate the corresponding future dates
last_date = ts.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps)

# Combine into a DataFrame
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted_High': forecast.values
})

print(forecast_df)

        Date  Predicted_High
0 2024-08-01       21.122835
1 2024-08-02       21.177180
2 2024-08-03       21.325083
3 2024-08-04       21.063214
4 2024-08-05       20.843164
5 2024-08-06       20.602638
6 2024-08-07       20.850528


Now that we have a prediction for the next seven days, we can compare it to the actual temperatures taken. We'll do this by merging this output with a query from the actual data.

In [8]:
met_data_actual=met_data_dublin.query("Temp_date>='2024-08-01' and Temp_date<='2024-08-07'")
met_data_actual=met_data_actual.drop(['Low','East','North'],axis=1)
result=met_data_actual.merge(forecast_df,left_on="Temp_date",right_on="Date")
result

,High,Temp_date,Location,Date,Predicted_High
0,23.2,2024-08-01,Dublin,2024-08-01,21.122835
1,22.9,2024-08-02,Dublin,2024-08-02,21.177180
2,20.4,2024-08-03,Dublin,2024-08-03,21.325083
3,23.5,2024-08-04,Dublin,2024-08-04,21.063214
4,24.5,2024-08-05,Dublin,2024-08-05,20.843164
5,21.8,2024-08-06,Dublin,2024-08-06,20.602638
6,20.0,2024-08-07,Dublin,2024-08-07,20.850528


We have our 1st result. The algorithm predictioned temperatures ranging from 20.6C to 21.3C. The actual temperatures range from 20.0 up to 23.5. Comparing these to the actual results, there are some variations with a temperature difference of almost 4C on the 5th. Let's continue on with the other 2 cities.

In [9]:
# Load and prepare data
met_data_query=met_data_waterford.query("Temp_date<='2024-07-31'")
met_data_query['Temp_date'] = pd.to_datetime(met_data_query['Temp_date'])
met_data_query = met_data_query.sort_values("Temp_date")

# Use day number as a feature
met_data_query['day_num'] = (met_data_query['Temp_date'] - met_data_query['Temp_date'].min()).dt.days

# Fit model
X = met_data_query[['day_num']]
y = met_data_query['High']
model = LinearRegression()
model.fit(X, y)

# Predict next 7 days
future_days = pd.DataFrame({'day_num': range(met_data_query['day_num'].max() + 1, met_data_query['day_num'].max() + 8)})
predictions = model.predict(future_days)

# Set datetime index
ts = met_data_query.set_index('Temp_date')['High']

# Fit model
model = SARIMAX(ts, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
results = model.fit()

# Forecast
forecast_steps = 7
forecast = results.forecast(steps=forecast_steps)

# Generate the corresponding future dates
last_date = ts.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps)

# Combine into a DataFrame
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted_High': forecast.values
})

print(forecast_df)

        Date  Predicted_High
0 2024-08-01       21.974165
1 2024-08-02       21.672167
2 2024-08-03       21.511252
3 2024-08-04       20.969952
4 2024-08-05       20.649404
5 2024-08-06       20.693058
6 2024-08-07       20.893744


In [10]:
met_data_actual=met_data_waterford.query("Temp_date>='2024-08-01' and Temp_date<='2024-08-07'")
met_data_actual=met_data_actual.drop(['Low','East','North'],axis=1)
result=met_data_actual.merge(forecast_df,left_on="Temp_date",right_on="Date")
result

,High,Temp_date,Location,Date,Predicted_High
0,23.7,2024-08-01,waterford,2024-08-01,21.974165
1,21.7,2024-08-02,waterford,2024-08-02,21.672167
2,20.2,2024-08-03,waterford,2024-08-03,21.511252
3,18.7,2024-08-04,waterford,2024-08-04,20.969952
4,19.5,2024-08-05,waterford,2024-08-05,20.649404
5,20.0,2024-08-06,waterford,2024-08-06,20.693058
6,19.0,2024-08-07,waterford,2024-08-07,20.893744


Now we have a result for Waterford. The predictions are in the range 20.6C up to 21.9C. The actual temperatures are in the range 18.7 up to 23.7. Again there are variances between the prediction and the actual results, but the differences not as high as those for Dublin and also on the 2nd and 5th, the predictions are actually quite accurate.

In [11]:
# Load and prepare data
met_data_query=met_data_letterkenny.query("Temp_date<='2024-07-31'")
met_data_query['Temp_date'] = pd.to_datetime(met_data_query['Temp_date'])
met_data_query = met_data_query.sort_values("Temp_date")

# Use day number as a feature
met_data_query['day_num'] = (met_data_query['Temp_date'] - met_data_query['Temp_date'].min()).dt.days

# Fit model
X = met_data_query[['day_num']]
y = met_data_query['High']
model = LinearRegression()
model.fit(X, y)

# Predict next 7 days
future_days = pd.DataFrame({'day_num': range(met_data_query['day_num'].max() + 1, met_data_query['day_num'].max() + 8)})
predictions = model.predict(future_days)

# Set datetime index
ts = met_data_query.set_index('Temp_date')['High']

# Fit model
model = SARIMAX(ts, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
results = model.fit()

# Forecast
forecast_steps = 7
forecast = results.forecast(steps=forecast_steps)

# Generate the corresponding future dates
last_date = ts.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps)

# Combine into a DataFrame
forecast_df = pd.DataFrame({
    'Date': future_dates,
    'Predicted_High': forecast.values
})

print(forecast_df)

        Date  Predicted_High
0 2024-08-01       20.976337
1 2024-08-02       20.659229
2 2024-08-03       20.459649
3 2024-08-04       20.033394
4 2024-08-05       19.634866
5 2024-08-06       19.491329
6 2024-08-07       19.908219


In [12]:
met_data_actual=met_data_letterkenny.query("Temp_date>='2024-08-01' and Temp_date<='2024-08-07'")
met_data_actual=met_data_actual.drop(['Low','East','North'],axis=1)
result=met_data_actual.merge(forecast_df,left_on="Temp_date",right_on="Date")
result

,High,Temp_date,Location,Date,Predicted_High
0,21.7,2024-08-01,letterkenny,2024-08-01,20.976337
1,18.0,2024-08-02,letterkenny,2024-08-02,20.659229
2,17.7,2024-08-03,letterkenny,2024-08-03,20.459649
3,20.4,2024-08-04,letterkenny,2024-08-04,20.033394
4,18.4,2024-08-05,letterkenny,2024-08-05,19.634866
5,17.6,2024-08-06,letterkenny,2024-08-06,19.491329
6,17.6,2024-08-07,letterkenny,2024-08-07,19.908219


Finally, the results for Letterkenny. The predictions are in the range of 20.9 down to 19.9. The actual temperature ranges are 17.6 up to 21.7. We have one instance of fair accuracy on the 4th when the prediction is within 4 decimal places of the actual.

In reality when it comes to temperature predictions, using previous temperature readings on the same day isn't really all that accurate as other factors are taken into account, most notably barometric pressure. However, the predictions do closely resemble the actual results and the variances are not huge.